### Demomstrates the performance of TT-Cross Versus NN for function approximation

In [ ]:
import torch
from tt_utils import *

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

def gmm(n=2,nmix=3,L=1,mx_coef=None,mu=None,s=0.1, device='cpu'):
    """
        Mixture of spherical Gaussians (un-normalized)
        nmix: number of mixture coefficients
        n: dimension of the domain
        s: variance
        mu: the centers assumed to be in : [-L,L]^n
    """
    n_sqrt = torch.sqrt(torch.tensor([n]).to(device))
    if mx_coef is None: # if centers and mixture coef are not given, generate them randomly
        mx_coef = torch.rand(nmix).to(device)
        mx_coef = mx_coef/torch.sum(mx_coef)
        mu = (torch.rand(nmix,n).to(device)-0.5)*2*L

    def pdf(x):
        result = torch.tensor([0]).to(device)
        for k in range(nmix):
            l = torch.linalg.norm(mu[k]-x, dim=1)/n_sqrt
            result = result + mx_coef[k]*torch.exp(-(l/s)**2)
        return result

    return pdf


In [ ]:
dim = 10
L = 1
nmix = 1
s = 0.2
pdf = gmm(n=dim,nmix=nmix,L=L,mx_coef=None,mu=None,s=s, device=device) # Or define an arbitrary function of your choice

##### Find TT approximation

In [ ]:
# So TT approximation
n_discretization = torch.tensor([200]*dim).to(device)
domain = [torch.linspace(-L,L,n_discretization[i]).to(device) for i in range(dim)] 

import time 
t1 = time.time()
tt_gmm = cross_approximate(fcn=pdf,  max_batch=10**6, domain=domain, 
                        rmax=200, nswp=20, eps=1e-3, verbose=True, 
                        kickrank=3, device=device)
t2 = time.time()
print("time taken: ", t2-t1)


#### Prepare test set and train set for NN


In [ ]:

ndata_train = 218400
ndata_test = 1000

x_train = 2*L*(-0.5 + torch.rand((ndata_train,dim)).to(device))
y_train = pdf(x_train)

x_test = 2*L*(-0.5 + torch.rand((ndata_test,dim)).to(device))
y_test = pdf(x_test)

data_train = torch.cat((x_train.view(-1,dim),y_train.view(-1,1)),dim=-1)
data_test = torch.cat((x_test.view(-1,dim),y_test.view(-1,1)),dim=-1)

In [ ]:
# Test the error in TT-approximation
y_tt =  get_value(tt_model=tt_gmm, x=x_test.to(device),  domain=domain, 
                    n_discretization=n_discretization , max_batch=10**5, device=device)

mse_tt = (((y_tt.view(-1)-y_test.view(-1))/(1e-9+y_test.view(-1).abs()))**2).mean()
print("mse_tt: ", mse_tt)

##### Fit an NN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class NeuralNetwork(nn.Module):
    def __init__(self, dim=2, width=32):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(dim, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, 1),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork(dim=dim, width=dim*nmix*10).to(device)
def train_loop(data, model, loss_fn, optimizer, batch_size):
    size = data.shape[0]
    counter = 0
    for i in range(int(size/batch_size)-1):
        # Compute prediction and loss
        next_counter = (counter+batch_size)
        x_data = data[counter:next_counter,:-1]
        y_data = data[counter:next_counter,-1].view(-1,1)
        y_pred = model(x_data)
        loss = loss_fn(y_pred, y_data)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        counter = 1*next_counter

        if (i % int(0.25*size/batch_size)) == 0 :
            loss = loss.item()
            print(f"loss: {loss:>7f}")


def test_loop(data, model, loss_fn):
    x_data = data[:,:-1]
    y_data = data[:,-1]
    with torch.no_grad():
        pred = model(x_data)
        test_loss = loss_fn(pred, y_data).item()
    print(f"Test Error: ", test_loss)

##### Train NN

In [ ]:
# Initialize NN
learning_rate = 1e-3
batch_size = 100
epochs = 10
y_nn_0 = model(x_test)
mse_nn_0 = (((y_nn_0.view(-1)-y_test.view(-1))/(1e-9+y_test.view(-1).abs()))**2).mean()
print("mse_nn_0: ", mse_nn_0)

# Train NN
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
batch_size = 100
epochs = 10
t1 = time.time()
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(data_train, model, loss_fn, optimizer, batch_size)
    test_loop(data_test, model, loss_fn)
t2 = time.time()
print("time taken: ", t2-t1)
print("Done!")

In [ ]:
y_nn = model(x_test)
mse_nn = (((y_nn.view(-1)-y_test.view(-1))/(1e-9+y_test.view(-1).abs()))**2).mean()
print("NN Relative MSE: ", mse_nn)

In [ ]:
mse_tt = (((y_tt.view(-1)-y_test.view(-1))/(1e-9+y_test.view(-1).abs()))**2).mean()
print("TT Relative MSE: ", mse_tt)

In [ ]:
y_nn = model(x_test)
mse_nn = ((y_nn-y_test)**2).mean()
print("NN Absolute MSE: ", mse_nn)

In [ ]:
mse_tt = (((y_tt.view(-1)-y_test.view(-1)))**2).mean()
print("TT Absolute MSE: ", mse_tt)